In [1]:
# ============================================================
# CELL 1: INSTRUCTION-GENERATION SETUP AND INPUT VALIDATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import re

import numpy as np
import pandas as pd


print("=" * 80)
print("NOTEBOOK 07: GROUNDED INSTRUCTION GENERATION")
print("=" * 80)


# ------------------------------------------------------------
# 1. Project directories
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DATA_DIR = (
    DATA_DIR
    / "processed"
)

ROAD_FLOOD_MASTER_DIR = (
    PROCESSED_DATA_DIR
    / "road_flood_grounding"
    / "batch_grounding"
    / "master"
)

NOTEBOOK_07_OUTPUT_DIR = (
    PROCESSED_DATA_DIR
    / "instruction_generation"
)

INSTRUCTION_RECORDS_DIR = (
    NOTEBOOK_07_OUTPUT_DIR
    / "instruction_records"
)

SCENE_INSTRUCTION_DIR = (
    NOTEBOOK_07_OUTPUT_DIR
    / "scene_instructions"
)

MASTER_INSTRUCTION_DIR = (
    NOTEBOOK_07_OUTPUT_DIR
    / "master"
)

VALIDATION_OUTPUT_DIR = (
    NOTEBOOK_07_OUTPUT_DIR
    / "validation"
)

for directory in [
    NOTEBOOK_07_OUTPUT_DIR,
    INSTRUCTION_RECORDS_DIR,
    SCENE_INSTRUCTION_DIR,
    MASTER_INSTRUCTION_DIR,
    VALIDATION_OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("\nDIRECTORIES")
print("-" * 80)

print(
    f"Project root             : "
    f"{PROJECT_ROOT}"
)

print(
    f"Notebook 06 master input : "
    f"{ROAD_FLOOD_MASTER_DIR}"
)

print(
    f"Notebook 07 output       : "
    f"{NOTEBOOK_07_OUTPUT_DIR}"
)


# ------------------------------------------------------------
# 2. Canonical Notebook 06 inputs
# ------------------------------------------------------------

COMBINED_GROUNDING_CSV = (
    ROAD_FLOOD_MASTER_DIR
    / "master_combined_transportation_flood_grounding.csv"
)

SCENE_PROFILES_CSV = (
    ROAD_FLOOD_MASTER_DIR
    / "master_road_flood_scene_profiles.csv"
)

PHYSICAL_EDGES_CSV = (
    ROAD_FLOOD_MASTER_DIR
    / "master_physical_edge_flood_grounding.csv"
)

PROVENANCE_JSON = (
    ROAD_FLOOD_MASTER_DIR
    / "road_flood_grounding_provenance.json"
)

NOTEBOOK_06_SUMMARY_JSON = (
    ROAD_FLOOD_MASTER_DIR
    / "notebook_06_completion_summary.json"
)


required_input_paths = {
    "combined_grounding": COMBINED_GROUNDING_CSV,
    "scene_profiles": SCENE_PROFILES_CSV,
    "physical_edges": PHYSICAL_EDGES_CSV,
    "provenance": PROVENANCE_JSON,
    "notebook_06_summary": NOTEBOOK_06_SUMMARY_JSON,
}


print("\nINPUT FILE VALIDATION")
print("-" * 80)

missing_input_files = []

for input_name, input_path in (
    required_input_paths.items()
):
    exists = input_path.exists()

    print(
        f"{input_name:<25}: "
        f"{exists} | {input_path}"
    )

    if not exists:
        missing_input_files.append(
            str(input_path)
        )

if missing_input_files:
    raise FileNotFoundError(
        "Required Notebook 06 files are missing:\n"
        + "\n".join(
            missing_input_files
        )
    )


# ------------------------------------------------------------
# 3. Load finalized Notebook 06 outputs
# ------------------------------------------------------------

combined_grounding_df = pd.read_csv(
    COMBINED_GROUNDING_CSV,
    keep_default_na=False,
)

scene_profiles_df = pd.read_csv(
    SCENE_PROFILES_CSV,
    keep_default_na=False,
)

physical_edges_df = pd.read_csv(
    PHYSICAL_EDGES_CSV,
    keep_default_na=False,
    low_memory=False,
)

with open(
    PROVENANCE_JSON,
    "r",
    encoding="utf-8",
) as file:
    grounding_provenance = json.load(
        file
    )

with open(
    NOTEBOOK_06_SUMMARY_JSON,
    "r",
    encoding="utf-8",
) as file:
    notebook_06_summary = json.load(
        file
    )


# ------------------------------------------------------------
# 3A. Normalize valid semantic "None" labels
# ------------------------------------------------------------

combined_grounding_df[
    "critical_network_disruption"
] = (
    combined_grounding_df[
        "critical_network_disruption"
    ]
    .astype(str)
    .str.strip()
    .replace(
        {
            "": "None",
            "nan": "None",
            "NaN": "None",
        }
    )
)

scene_profiles_df[
    "critical_network_disruption"
] = (
    scene_profiles_df[
        "critical_network_disruption"
    ]
    .astype(str)
    .str.strip()
    .replace(
        {
            "": "None",
            "nan": "None",
            "NaN": "None",
        }
    )
)


print("\nLOADED NOTEBOOK 06 OUTPUTS")
print("-" * 80)

print(
    f"Combined grounding scenes : "
    f"{len(combined_grounding_df):,}"
)

print(
    f"Scene profiles            : "
    f"{len(scene_profiles_df):,}"
)

print(
    f"Physical-edge records     : "
    f"{len(physical_edges_df):,}"
)

print(
    f"Grounding version         : "
    f"{grounding_provenance.get('grounding_version')}"
)

print(
    f"Notebook 06 status        : "
    f"{notebook_06_summary.get('status')}"
)


# ------------------------------------------------------------
# 4. Normalize scene IDs
# ------------------------------------------------------------

for dataframe in [
    combined_grounding_df,
    scene_profiles_df,
    physical_edges_df,
]:
    dataframe[
        "scene_id"
    ] = (
        dataframe[
            "scene_id"
        ]
        .astype(str)
        .str.strip()
    )


# ------------------------------------------------------------
# 5. Validate required scene-level grounding fields
# ------------------------------------------------------------

required_combined_columns = [
    # Identity
    "scene_id",

    # Road-network coverage
    "physical_edge_count",
    "edges_with_valid_coverage_count",
    "raster_coverage_share",

    # Flood exposure
    "event_flood_edge_count",
    "event_flood_edge_share",
    "low_flood_exposure_edge_count",
    "moderate_flood_exposure_edge_count",
    "high_flood_exposure_edge_count",

    # Critical transportation exposure
    "critical_edge_count",
    "flooded_critical_edge_count",
    "flooded_critical_edge_share",

    "critical_low_redundancy_count",
    "flooded_critical_low_redundancy_count",
    "flooded_critical_low_redundancy_share",

    "physical_bridge_count",
    "flooded_physical_bridge_count",

    "bridge_bottleneck_count",
    "flooded_bridge_bottleneck_count",

    "major_road_bottleneck_count",
    "flooded_major_road_bottleneck_count",

    # Semantic classes
    "scene_flood_burden",
    "critical_network_disruption",

    # Ranking
    "transportation_flood_disruption_score",
    "transportation_flood_disruption_rank",

    # Reliability
    "grounding_quality_warning",
    "grounding_reliability",

    # Grounded tokens
    "transportation_knowledge_token",
    "scene_road_flood_token",
    "combined_transportation_flood_token",

    # Provenance
    "grounding_version",
    "flood_definition",
    "scene_interpretation_note",
]

missing_combined_columns = [
    column
    for column in required_combined_columns
    if column not in combined_grounding_df.columns
]


print("\nREQUIRED COLUMN VALIDATION")
print("-" * 80)

print(
    f"Required columns       : "
    f"{len(required_combined_columns):,}"
)

print(
    f"Missing columns        : "
    f"{len(missing_combined_columns):,}"
)

print(
    f"Missing column names   : "
    f"{missing_combined_columns}"
)

if missing_combined_columns:
    raise ValueError(
        "Combined grounding dataset is missing required "
        f"columns: {missing_combined_columns}"
    )


# ------------------------------------------------------------
# 6. Validate scene alignment
# ------------------------------------------------------------

combined_scene_ids = set(
    combined_grounding_df[
        "scene_id"
    ]
)

profile_scene_ids = set(
    scene_profiles_df[
        "scene_id"
    ]
)

physical_edge_scene_ids = set(
    physical_edges_df[
        "scene_id"
    ]
)


scene_alignment_checks = {
    "combined_matches_profiles": (
        combined_scene_ids
        == profile_scene_ids
    ),
    "combined_matches_physical_edges": (
        combined_scene_ids
        == physical_edge_scene_ids
    ),
    "combined_scene_ids_unique": (
        combined_grounding_df[
            "scene_id"
        ].duplicated().sum()
        == 0
    ),
    "profile_scene_ids_unique": (
        scene_profiles_df[
            "scene_id"
        ].duplicated().sum()
        == 0
    ),
}


print("\nSCENE ALIGNMENT")
print("-" * 80)

for check_name, check_result in (
    scene_alignment_checks.items()
):
    print(
        f"{check_name:<35}: "
        f"{check_result}"
    )

if not all(
    scene_alignment_checks.values()
):
    raise ValueError(
        "Notebook 06 scene alignment validation failed."
    )


# ------------------------------------------------------------
# 7. Check missing values in instruction-critical fields
# ------------------------------------------------------------

instruction_critical_columns = [
    "scene_id",
    "physical_edge_count",
    "event_flood_edge_count",
    "event_flood_edge_share",
    "critical_edge_count",
    "flooded_critical_edge_count",
    "scene_flood_burden",
    "critical_network_disruption",
    "transportation_flood_disruption_score",
    "transportation_flood_disruption_rank",
    "grounding_reliability",
    "combined_transportation_flood_token",
]

missing_value_summary_df = pd.DataFrame(
    {
        "column": instruction_critical_columns,
        "missing_count": [
            int(
                combined_grounding_df[
                    column
                ].isna().sum()
            )
            for column in instruction_critical_columns
        ],
    }
)

missing_value_summary_df[
    "complete"
] = (
    missing_value_summary_df[
        "missing_count"
    ] == 0
)


print("\nMISSING-VALUE VALIDATION")
print("-" * 80)

display(
    missing_value_summary_df
)

if not missing_value_summary_df[
    "complete"
].all():
    raise ValueError(
        "Instruction-critical fields contain missing values."
    )


# ------------------------------------------------------------
# 8. Validate allowed semantic labels
# ------------------------------------------------------------

allowed_flood_burden_labels = {
    "Minimal",
    "Low",
    "Moderate",
    "High",
}

allowed_disruption_labels = {
    "None",
    "Low",
    "Moderate",
    "High",
}

allowed_reliability_labels = {
    "Low",
    "Moderate",
    "High",
}


observed_flood_burden_labels = set(
    combined_grounding_df[
        "scene_flood_burden"
    ].dropna().astype(str)
)

observed_disruption_labels = set(
    combined_grounding_df[
        "critical_network_disruption"
    ].dropna().astype(str)
)

observed_reliability_labels = set(
    combined_grounding_df[
        "grounding_reliability"
    ].dropna().astype(str)
)


invalid_flood_burden_labels = (
    observed_flood_burden_labels
    - allowed_flood_burden_labels
)

invalid_disruption_labels = (
    observed_disruption_labels
    - allowed_disruption_labels
)

invalid_reliability_labels = (
    observed_reliability_labels
    - allowed_reliability_labels
)


print("\nSEMANTIC LABEL VALIDATION")
print("-" * 80)

print(
    f"Flood burden labels observed : "
    f"{sorted(observed_flood_burden_labels)}"
)

print(
    f"Disruption labels observed    : "
    f"{sorted(observed_disruption_labels)}"
)

print(
    f"Reliability labels observed   : "
    f"{sorted(observed_reliability_labels)}"
)

print(
    f"Invalid flood labels          : "
    f"{sorted(invalid_flood_burden_labels)}"
)

print(
    f"Invalid disruption labels     : "
    f"{sorted(invalid_disruption_labels)}"
)

print(
    f"Invalid reliability labels    : "
    f"{sorted(invalid_reliability_labels)}"
)

if (
    invalid_flood_burden_labels
    or invalid_disruption_labels
    or invalid_reliability_labels
):
    raise ValueError(
        "Unexpected semantic class labels were detected."
    )


# ------------------------------------------------------------
# 9. Review class distributions
# ------------------------------------------------------------

flood_burden_distribution_df = (
    combined_grounding_df[
        "scene_flood_burden"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "scene_flood_burden"
    )
    .reset_index(
        name="scene_count"
    )
)

flood_burden_distribution_df[
    "scene_share"
] = (
    flood_burden_distribution_df[
        "scene_count"
    ]
    / len(
        combined_grounding_df
    )
).round(4)


disruption_distribution_df = (
    combined_grounding_df[
        "critical_network_disruption"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "critical_network_disruption"
    )
    .reset_index(
        name="scene_count"
    )
)

disruption_distribution_df[
    "scene_share"
] = (
    disruption_distribution_df[
        "scene_count"
    ]
    / len(
        combined_grounding_df
    )
).round(4)


reliability_distribution_df = (
    combined_grounding_df[
        "grounding_reliability"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "grounding_reliability"
    )
    .reset_index(
        name="scene_count"
    )
)

reliability_distribution_df[
    "scene_share"
] = (
    reliability_distribution_df[
        "scene_count"
    ]
    / len(
        combined_grounding_df
    )
).round(4)


print("\nSCENE FLOOD-BURDEN DISTRIBUTION")
print("-" * 80)

display(
    flood_burden_distribution_df
)


print("\nCRITICAL-DISRUPTION DISTRIBUTION")
print("-" * 80)

display(
    disruption_distribution_df
)


print("\nGROUNDING-RELIABILITY DISTRIBUTION")
print("-" * 80)

display(
    reliability_distribution_df
)


# ------------------------------------------------------------
# 10. Prepare physical-edge summaries for instruction creation
# ------------------------------------------------------------

required_physical_edge_columns = [
    "scene_id",
    "physical_edge_id",
    "road_flood_exposure_class",
    "event_flood_fraction",
    "touches_event_flood",
    "is_critical_transport_edge",
    "is_critical_low_redundancy",
    "is_physical_bridge",
    "is_bridge_bottleneck",
    "is_major_road_bottleneck",
]

missing_physical_edge_columns = [
    column
    for column in required_physical_edge_columns
    if column not in physical_edges_df.columns
]


print("\nPHYSICAL-EDGE FIELD VALIDATION")
print("-" * 80)

print(
    f"Missing physical-edge fields: "
    f"{missing_physical_edge_columns}"
)

if missing_physical_edge_columns:
    raise ValueError(
        "Physical-edge grounding dataset is missing required "
        f"columns: {missing_physical_edge_columns}"
    )


# Normalize Boolean-like columns loaded from CSV.
boolean_physical_edge_columns = [
    "touches_event_flood",
    "is_critical_transport_edge",
    "is_critical_low_redundancy",
    "is_physical_bridge",
    "is_bridge_bottleneck",
    "is_major_road_bottleneck",
]


def normalize_boolean_series(series):
    """
    Normalize Boolean, numeric, and common string representations.
    """

    if pd.api.types.is_bool_dtype(
        series
    ):
        return (
            series
            .fillna(False)
            .astype(bool)
        )

    if pd.api.types.is_numeric_dtype(
        series
    ):
        return (
            pd.to_numeric(
                series,
                errors="coerce",
            )
            .fillna(0)
            .ne(0)
        )

    normalized_series = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return normalized_series.isin(
        {
            "true",
            "1",
            "yes",
            "y",
        }
    )


for column in (
    boolean_physical_edge_columns
):
    physical_edges_df[
        column
    ] = normalize_boolean_series(
        physical_edges_df[
            column
        ]
    )


# ------------------------------------------------------------
# 11. Create scene-level edge evidence summaries
# ------------------------------------------------------------

def summarize_scene_edge_evidence(
    scene_edges_df,
):
    """
    Build a compact evidence summary from physical road edges.
    """

    flooded_edges_df = (
        scene_edges_df.loc[
            scene_edges_df[
                "touches_event_flood"
            ]
        ]
        .copy()
    )

    highly_exposed_edges_df = (
        scene_edges_df.loc[
            scene_edges_df[
                "road_flood_exposure_class"
            ]
            == "High Flood Exposure"
        ]
        .copy()
    )

    flooded_critical_edges_df = (
        scene_edges_df.loc[
            scene_edges_df[
                "touches_event_flood"
            ]
            &
            scene_edges_df[
                "is_critical_transport_edge"
            ]
        ]
        .copy()
    )

    flooded_low_redundancy_edges_df = (
        scene_edges_df.loc[
            scene_edges_df[
                "touches_event_flood"
            ]
            &
            scene_edges_df[
                "is_critical_low_redundancy"
            ]
        ]
        .copy()
    )

    flooded_bridge_edges_df = (
        scene_edges_df.loc[
            scene_edges_df[
                "touches_event_flood"
            ]
            &
            scene_edges_df[
                "is_physical_bridge"
            ]
        ]
        .copy()
    )

    flooded_bottleneck_edges_df = (
        scene_edges_df.loc[
            scene_edges_df[
                "touches_event_flood"
            ]
            &
            (
                scene_edges_df[
                    "is_bridge_bottleneck"
                ]
                |
                scene_edges_df[
                    "is_major_road_bottleneck"
                ]
            )
        ]
        .copy()
    )

    top_exposed_edges_df = (
        scene_edges_df
        .sort_values(
            "event_flood_fraction",
            ascending=False,
        )
        .head(5)
    )

    top_edge_records = []

    for _, edge_row in (
        top_exposed_edges_df.iterrows()
    ):
        top_edge_records.append(
            {
                "physical_edge_id": str(
                    edge_row[
                        "physical_edge_id"
                    ]
                ),
                "event_flood_fraction": round(
                    float(
                        edge_row[
                            "event_flood_fraction"
                        ]
                    ),
                    4,
                ),
                "exposure_class": str(
                    edge_row[
                        "road_flood_exposure_class"
                    ]
                ),
                "critical_transport_edge": bool(
                    edge_row[
                        "is_critical_transport_edge"
                    ]
                ),
                "critical_low_redundancy": bool(
                    edge_row[
                        "is_critical_low_redundancy"
                    ]
                ),
                "physical_bridge": bool(
                    edge_row[
                        "is_physical_bridge"
                    ]
                ),
            }
        )

    return {
        "physical_edge_count": int(
            len(
                scene_edges_df
            )
        ),
        "flooded_edge_count": int(
            len(
                flooded_edges_df
            )
        ),
        "high_exposure_edge_count": int(
            len(
                highly_exposed_edges_df
            )
        ),
        "flooded_critical_edge_count": int(
            len(
                flooded_critical_edges_df
            )
        ),
        "flooded_low_redundancy_edge_count": int(
            len(
                flooded_low_redundancy_edges_df
            )
        ),
        "flooded_bridge_edge_count": int(
            len(
                flooded_bridge_edges_df
            )
        ),
        "flooded_bottleneck_edge_count": int(
            len(
                flooded_bottleneck_edges_df
            )
        ),
        "maximum_event_flood_fraction": round(
            float(
                pd.to_numeric(
                    scene_edges_df[
                        "event_flood_fraction"
                    ],
                    errors="coerce",
                )
                .fillna(0)
                .max()
            ),
            4,
        ),
        "top_exposed_edges": (
            top_edge_records
        ),
    }


scene_edge_evidence_records = []

for scene_id, scene_edges_df in (
    physical_edges_df.groupby(
        "scene_id"
    )
):
    evidence_record = (
        summarize_scene_edge_evidence(
            scene_edges_df
        )
    )

    evidence_record[
        "scene_id"
    ] = str(
        scene_id
    )

    scene_edge_evidence_records.append(
        evidence_record
    )


scene_edge_evidence_df = pd.DataFrame(
    scene_edge_evidence_records
)


print("\nSCENE EDGE-EVIDENCE SUMMARY")
print("-" * 80)

print(
    f"Evidence records generated: "
    f"{len(scene_edge_evidence_df):,}"
)

print(
    f"Duplicate scene evidence  : "
    f"{scene_edge_evidence_df['scene_id'].duplicated().sum():,}"
)


# ------------------------------------------------------------
# 12. Merge instruction-generation source table
# ------------------------------------------------------------

instruction_source_df = (
    combined_grounding_df
    .merge(
        scene_edge_evidence_df,
        on="scene_id",
        how="left",
        validate="one_to_one",
        suffixes=(
            "",
            "_edge_evidence",
        ),
    )
    .sort_values(
        [
            "transportation_flood_disruption_rank",
            "scene_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


instruction_source_df[
    "instruction_source_ready"
] = (
    instruction_source_df[
        "combined_transportation_flood_token"
    ].notna()
    &
    instruction_source_df[
        "scene_flood_burden"
    ].notna()
    &
    instruction_source_df[
        "critical_network_disruption"
    ].notna()
    &
    instruction_source_df[
        "grounding_reliability"
    ].notna()
)


print("\nINSTRUCTION SOURCE READINESS")
print("-" * 80)

print(
    f"Instruction source scenes : "
    f"{len(instruction_source_df):,}"
)

print(
    f"Ready scenes              : "
    f"{instruction_source_df['instruction_source_ready'].sum():,}"
)

print(
    f"Not-ready scenes          : "
    f"{(~instruction_source_df['instruction_source_ready']).sum():,}"
)

if not instruction_source_df[
    "instruction_source_ready"
].all():
    raise ValueError(
        "One or more scenes are not ready for instruction generation."
    )


# ------------------------------------------------------------
# 13. Define Notebook 07 generation metadata
# ------------------------------------------------------------

INSTRUCTION_DATASET_VERSION = "v1.0"

INSTRUCTION_GENERATION_MODE = (
    "deterministic_grounded_templates"
)

INSTRUCTION_GENERATED_UTC = (
    datetime.now(
        timezone.utc
    )
    .replace(
        microsecond=0
    )
    .isoformat()
)

SUPPORTED_TASK_FAMILIES = [
    "scene_flood_assessment",
    "critical_network_assessment",
    "evidence_explanation",
    "transportation_prioritization",
    "reliability_aware_assessment",
    "structured_grounding_summary",
]


instruction_generation_metadata = {
    "notebook": (
        "07_instruction_generation"
    ),
    "instruction_dataset_version": (
        INSTRUCTION_DATASET_VERSION
    ),
    "source_grounding_version": (
        grounding_provenance.get(
            "grounding_version"
        )
    ),
    "generation_mode": (
        INSTRUCTION_GENERATION_MODE
    ),
    "generated_utc": (
        INSTRUCTION_GENERATED_UTC
    ),
    "scene_count": int(
        len(
            instruction_source_df
        )
    ),
    "physical_edge_record_count": int(
        len(
            physical_edges_df
        )
    ),
    "supported_task_families": (
        SUPPORTED_TASK_FAMILIES
    ),
    "source_file": str(
        COMBINED_GROUNDING_CSV
    ),
    "physical_edge_source_file": str(
        PHYSICAL_EDGES_CSV
    ),
    "grounding_definition": (
        grounding_provenance.get(
            "flood_definition"
        )
    ),
}


# ------------------------------------------------------------
# 14. Save validated instruction source table
# ------------------------------------------------------------

INSTRUCTION_SOURCE_CSV = (
    MASTER_INSTRUCTION_DIR
    / "instruction_generation_source.csv"
)

INSTRUCTION_SOURCE_JSON = (
    MASTER_INSTRUCTION_DIR
    / "instruction_generation_source.json"
)

INSTRUCTION_METADATA_JSON = (
    MASTER_INSTRUCTION_DIR
    / "instruction_generation_metadata.json"
)


instruction_source_df.to_csv(
    INSTRUCTION_SOURCE_CSV,
    index=False,
)


def make_json_safe(value):
    """
    Convert common NumPy and pandas objects to JSON-safe values.
    """

    if value is None:
        return None

    if isinstance(
        value,
        np.integer,
    ):
        return int(
            value
        )

    if isinstance(
        value,
        np.floating,
    ):
        if np.isnan(
            value
        ):
            return None

        return float(
            value
        )

    if isinstance(
        value,
        np.bool_,
    ):
        return bool(
            value
        )

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    if isinstance(
        value,
        (
            list,
            dict,
            str,
            int,
            float,
            bool,
        ),
    ):
        return value

    if pd.isna(
        value
    ):
        return None

    return str(
        value
    )


instruction_source_json_records = []

for record in instruction_source_df.to_dict(
    orient="records"
):
    safe_record = {}

    for key, value in record.items():
        safe_record[
            key
        ] = make_json_safe(
            value
        )

    instruction_source_json_records.append(
        safe_record
    )


with open(
    INSTRUCTION_SOURCE_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        instruction_source_json_records,
        file,
        indent=2,
        ensure_ascii=False,
    )


with open(
    INSTRUCTION_METADATA_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        instruction_generation_metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 15. Save validation tables
# ------------------------------------------------------------

MISSING_VALUE_VALIDATION_CSV = (
    VALIDATION_OUTPUT_DIR
    / "instruction_source_missing_value_validation.csv"
)

FLOOD_CLASS_DISTRIBUTION_CSV = (
    VALIDATION_OUTPUT_DIR
    / "scene_flood_burden_distribution.csv"
)

DISRUPTION_CLASS_DISTRIBUTION_CSV = (
    VALIDATION_OUTPUT_DIR
    / "critical_disruption_distribution.csv"
)

RELIABILITY_DISTRIBUTION_CSV = (
    VALIDATION_OUTPUT_DIR
    / "grounding_reliability_distribution.csv"
)


missing_value_summary_df.to_csv(
    MISSING_VALUE_VALIDATION_CSV,
    index=False,
)

flood_burden_distribution_df.to_csv(
    FLOOD_CLASS_DISTRIBUTION_CSV,
    index=False,
)

disruption_distribution_df.to_csv(
    DISRUPTION_CLASS_DISTRIBUTION_CSV,
    index=False,
)

reliability_distribution_df.to_csv(
    RELIABILITY_DISTRIBUTION_CSV,
    index=False,
)


# ------------------------------------------------------------
# 16. Final setup report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NOTEBOOK 07 SETUP COMPLETE")
print("=" * 80)

print(
    f"Instruction dataset version : "
    f"{INSTRUCTION_DATASET_VERSION}"
)

print(
    f"Grounding source version    : "
    f"{grounding_provenance.get('grounding_version')}"
)

print(
    f"Scenes ready                : "
    f"{len(instruction_source_df):,}"
)

print(
    f"Physical-edge records       : "
    f"{len(physical_edges_df):,}"
)

print(
    f"Supported task families     : "
    f"{len(SUPPORTED_TASK_FAMILIES):,}"
)

print(
    f"High-reliability scenes     : "
    f"{(instruction_source_df['grounding_reliability'] == 'High').sum():,}"
)

print(
    f"Moderate-reliability scenes : "
    f"{(instruction_source_df['grounding_reliability'] == 'Moderate').sum():,}"
)

print(
    f"Low-reliability scenes      : "
    f"{(instruction_source_df['grounding_reliability'] == 'Low').sum():,}"
)

print("\nOUTPUT FILES")
print("-" * 80)

for output_path in [
    INSTRUCTION_SOURCE_CSV,
    INSTRUCTION_SOURCE_JSON,
    INSTRUCTION_METADATA_JSON,
    MISSING_VALUE_VALIDATION_CSV,
    FLOOD_CLASS_DISTRIBUTION_CSV,
    DISRUPTION_CLASS_DISTRIBUTION_CSV,
    RELIABILITY_DISTRIBUTION_CSV,
]:
    print(
        output_path
    )


print("\nINSTRUCTION SOURCE PREVIEW")
print("-" * 80)

display(
    instruction_source_df[
        [
            "scene_id",
            "scene_flood_burden",
            "critical_network_disruption",
            "event_flood_edge_count",
            "flooded_critical_edge_count",
            "transportation_flood_disruption_rank",
            "grounding_reliability",
            "instruction_source_ready",
        ]
    ]
    .head(
        25
    )
)

print("=" * 80)

NOTEBOOK 07: GROUNDED INSTRUCTION GENERATION

DIRECTORIES
--------------------------------------------------------------------------------
Project root             : /home/adjeiowusu1/myproject/ResilientVLM
Notebook 06 master input : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/master
Notebook 07 output       : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/instruction_generation

INPUT FILE VALIDATION
--------------------------------------------------------------------------------
combined_grounding       : True | /home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/master/master_combined_transportation_flood_grounding.csv
scene_profiles           : True | /home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/master/master_road_flood_scene_profiles.csv
physical_edges           : True | /home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood

,column,missing_count,complete
0,scene_id,0,True
1,physical_edge_count,0,True
2,event_flood_edge_count,0,True
3,event_flood_edge_share,0,True
4,critical_edge_count,0,True
5,flooded_critical_edge_count,0,True
6,scene_flood_burden,0,True
7,critical_network_disruption,0,True
8,transportation_flood_disruption_score,0,True
9,transportation_flood_disruption_rank,0,True



SEMANTIC LABEL VALIDATION
--------------------------------------------------------------------------------
Flood burden labels observed : ['High', 'Low', 'Minimal', 'Moderate']
Disruption labels observed    : ['High', 'Low', 'Moderate', 'None']
Reliability labels observed   : ['High', 'Moderate']
Invalid flood labels          : []
Invalid disruption labels     : []
Invalid reliability labels    : []

SCENE FLOOD-BURDEN DISTRIBUTION
--------------------------------------------------------------------------------


,scene_flood_burden,scene_count,scene_share
0,High,14,0.5385
1,Moderate,7,0.2692
2,Low,4,0.1538
3,Minimal,1,0.0385



CRITICAL-DISRUPTION DISTRIBUTION
--------------------------------------------------------------------------------


...


GROUNDING-RELIABILITY DISTRIBUTION
--------------------------------------------------------------------------------


,grounding_reliability,scene_count,scene_share
0,High,19,0.7308
1,Moderate,7,0.2692



PHYSICAL-EDGE FIELD VALIDATION
--------------------------------------------------------------------------------
Missing physical-edge fields: []



SCENE EDGE-EVIDENCE SUMMARY
--------------------------------------------------------------------------------
Evidence records generated: 26
Duplicate scene evidence  : 0

INSTRUCTION SOURCE READINESS
--------------------------------------------------------------------------------
Instruction source scenes : 26
Ready scenes              : 26
Not-ready scenes          : 0

NOTEBOOK 07 SETUP COMPLETE
Instruction dataset version : v1.0
Grounding source version    : v1.1
Scenes ready                : 26
Physical-edge records       : 6,093
Supported task families     : 6
High-reliability scenes     : 19
Moderate-reliability scenes : 7
Low-reliability scenes      : 0

OUTPUT FILES
--------------------------------------------------------------------------------
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/instruction_generation/master/instruction_generation_source.csv
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/instruction_generation/master/instruction_generation_source

,scene_id,scene_flood_burden,critical_network_disruption,event_flood_edge_count,flooded_critical_edge_count,transportation_flood_disruption_rank,grounding_reliability,instruction_source_ready
0,Spain_7370579,Moderate,High,175,63,1,High,True
1,Spain_1167260,Moderate,High,129,20,2,High,True
2,Spain_8565131,Moderate,Moderate,53,20,3,High,True
3,Somalia_970508,High,Moderate,69,24,4,Moderate,True
4,India_500266,High,High,32,6,5,High,True
5,India_1050276,Moderate,High,17,5,6,High,True
6,India_956930,High,High,2,2,7,Moderate,True
7,India_1018327,High,High,7,3,8,High,True
8,India_383430,High,High,6,0,9,High,True
9,Nigeria_598959,High,Moderate,6,1,10,Moderate,True


In [2]:
# ============================================================
# CELL 2: GENERATE GROUNDED INSTRUCTION–RESPONSE RECORDS
# ============================================================

from hashlib import sha256


print("=" * 80)
print("GENERATING GROUNDED INSTRUCTION–RESPONSE RECORDS")
print("=" * 80)


# ------------------------------------------------------------
# 1. Instruction-generation configuration
# ------------------------------------------------------------

EXPECTED_SCENE_COUNT = len(
    instruction_source_df
)

EXPECTED_TASK_FAMILY_COUNT = len(
    SUPPORTED_TASK_FAMILIES
)

EXPECTED_INSTRUCTION_COUNT = (
    EXPECTED_SCENE_COUNT
    * EXPECTED_TASK_FAMILY_COUNT
)


TASK_FAMILY_DESCRIPTIONS = {
    "scene_flood_assessment": (
        "Assess the overall roadway flood burden in the scene."
    ),
    "critical_network_assessment": (
        "Assess whether flooding affects transportation-critical "
        "or low-redundancy roadway elements."
    ),
    "evidence_explanation": (
        "Explain the numerical and network evidence supporting "
        "the assigned flood and disruption classes."
    ),
    "transportation_prioritization": (
        "Determine the transportation-response priority using "
        "flood exposure, network criticality, and disruption rank."
    ),
    "reliability_aware_assessment": (
        "Provide a flood-disruption assessment while explicitly "
        "accounting for grounding reliability and known limitations."
    ),
    "structured_grounding_summary": (
        "Produce a compact structured summary of the scene-level "
        "road-flood grounding evidence."
    ),
}


# ------------------------------------------------------------
# 2. General helper functions
# ------------------------------------------------------------

def safe_int(
    value,
    default=0,
):
    """
    Convert a value to an integer safely.
    """

    try:
        if pd.isna(
            value
        ):
            return int(
                default
            )

        return int(
            round(
                float(
                    value
                )
            )
        )

    except (
        TypeError,
        ValueError,
    ):
        return int(
            default
        )


def safe_float(
    value,
    default=0.0,
):
    """
    Convert a value to a finite float safely.
    """

    try:
        numeric_value = float(
            value
        )

        if not np.isfinite(
            numeric_value
        ):
            return float(
                default
            )

        return numeric_value

    except (
        TypeError,
        ValueError,
    ):
        return float(
            default
        )


def format_count(
    value,
):
    """
    Format a numeric count using thousands separators.
    """

    return f"{safe_int(value):,}"


def format_share(
    value,
    decimals=1,
):
    """
    Convert a proportion to a percentage string.
    """

    percentage = (
        safe_float(
            value
        )
        * 100
    )

    return (
        f"{percentage:.{decimals}f}%"
    )


def normalize_optional_text(
    value,
):
    """
    Normalize optional textual fields.
    """

    if value is None:
        return ""

    text = str(
        value
    ).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "null",
    }:
        return ""

    return text


def clean_sentence_spacing(
    text,
):
    """
    Normalize whitespace while preserving readable sentences.
    """

    text = re.sub(
        r"\s+",
        " ",
        str(
            text
        ),
    )

    return text.strip()


def determine_priority_label(
    rank,
    scene_count,
    disruption_label,
):
    """
    Assign a deterministic response-priority category.
    """

    rank = safe_int(
        rank,
        default=scene_count,
    )

    if (
        rank
        <= max(
            1,
            int(
                np.ceil(
                    scene_count
                    * 0.20
                )
            ),
        )
        or disruption_label == "High"
    ):
        return "High"

    if (
        rank
        <= max(
            1,
            int(
                np.ceil(
                    scene_count
                    * 0.60
                )
            ),
        )
        or disruption_label == "Moderate"
    ):
        return "Moderate"

    return "Lower"


def build_reliability_statement(
    scene_row,
):
    """
    Generate reliability-aware interpretation language.
    """

    reliability = str(
        scene_row[
            "grounding_reliability"
        ]
    ).strip()

    warning = normalize_optional_text(
        scene_row.get(
            "grounding_quality_warning",
            "",
        )
    )

    coverage_share = safe_float(
        scene_row[
            "raster_coverage_share"
        ]
    )

    if reliability == "High":
        statement = (
            "The grounding reliability is high, so the reported "
            "road-flood relationships can be interpreted with relatively "
            "strong confidence within the mapped scene."
        )

    elif reliability == "Moderate":
        statement = (
            "The grounding reliability is moderate, so the findings "
            "should be interpreted with caution and confirmed with "
            "additional scene or network evidence when used for decisions."
        )

    else:
        statement = (
            "The grounding reliability is low, so the reported findings "
            "should be treated as preliminary rather than definitive."
        )

    statement += (
        f" Road-raster coverage is {coverage_share:.1%}."
    )

    if warning:
        statement += (
            f" The recorded quality warning is: {warning}."
        )

    return clean_sentence_spacing(
        statement
    )


def build_top_edge_evidence_text(
    top_exposed_edges,
    maximum_edges=3,
):
    """
    Convert the top exposed-edge list to compact explanatory text.
    """

    if not isinstance(
        top_exposed_edges,
        list,
    ):
        return (
            "No ranked physical-edge evidence was available."
        )

    if len(
        top_exposed_edges
    ) == 0:
        return (
            "No exposed physical roadway edges were identified."
        )

    edge_descriptions = []

    for edge_record in (
        top_exposed_edges[
            :maximum_edges
        ]
    ):
        edge_id = str(
            edge_record.get(
                "physical_edge_id",
                "unknown",
            )
        )

        flood_fraction = safe_float(
            edge_record.get(
                "event_flood_fraction",
                0,
            )
        )

        exposure_class = str(
            edge_record.get(
                "exposure_class",
                "Unknown",
            )
        )

        characteristics = []

        if edge_record.get(
            "critical_transport_edge",
            False,
        ):
            characteristics.append(
                "critical"
            )

        if edge_record.get(
            "critical_low_redundancy",
            False,
        ):
            characteristics.append(
                "low-redundancy"
            )

        if edge_record.get(
            "physical_bridge",
            False,
        ):
            characteristics.append(
                "bridge"
            )

        if characteristics:
            characteristic_text = (
                ", ".join(
                    characteristics
                )
            )

            descriptor = (
                f"{edge_id}: {flood_fraction:.1%} flooded, "
                f"{exposure_class}, {characteristic_text}"
            )

        else:
            descriptor = (
                f"{edge_id}: {flood_fraction:.1%} flooded, "
                f"{exposure_class}"
            )

        edge_descriptions.append(
            descriptor
        )

    return (
        "Top exposed edges include "
        + "; ".join(
            edge_descriptions
        )
        + "."
    )


def create_instruction_id(
    scene_id,
    task_family,
):
    """
    Create a stable deterministic instruction identifier.
    """

    identifier_source = (
        f"{INSTRUCTION_DATASET_VERSION}"
        f"|{scene_id}"
        f"|{task_family}"
    )

    identifier_hash = sha256(
        identifier_source.encode(
            "utf-8"
        )
    ).hexdigest()[
        :12
    ]

    return (
        f"instruction_{identifier_hash}"
    )


# ------------------------------------------------------------
# 3. Scene evidence extraction
# ------------------------------------------------------------

def extract_scene_evidence(
    scene_row,
):
    """
    Extract and normalize evidence used across all task families.
    """

    physical_edge_count = safe_int(
        scene_row[
            "physical_edge_count"
        ]
    )

    covered_edge_count = safe_int(
        scene_row[
            "edges_with_valid_coverage_count"
        ]
    )

    flooded_edge_count = safe_int(
        scene_row[
            "event_flood_edge_count"
        ]
    )

    flooded_edge_share = safe_float(
        scene_row[
            "event_flood_edge_share"
        ]
    )

    critical_edge_count = safe_int(
        scene_row[
            "critical_edge_count"
        ]
    )

    flooded_critical_edge_count = safe_int(
        scene_row[
            "flooded_critical_edge_count"
        ]
    )

    flooded_critical_edge_share = safe_float(
        scene_row[
            "flooded_critical_edge_share"
        ]
    )

    low_redundancy_count = safe_int(
        scene_row[
            "critical_low_redundancy_count"
        ]
    )

    flooded_low_redundancy_count = safe_int(
        scene_row[
            "flooded_critical_low_redundancy_count"
        ]
    )

    physical_bridge_count = safe_int(
        scene_row[
            "physical_bridge_count"
        ]
    )

    flooded_bridge_count = safe_int(
        scene_row[
            "flooded_physical_bridge_count"
        ]
    )

    bridge_bottleneck_count = safe_int(
        scene_row[
            "bridge_bottleneck_count"
        ]
    )

    flooded_bridge_bottleneck_count = safe_int(
        scene_row[
            "flooded_bridge_bottleneck_count"
        ]
    )

    major_bottleneck_count = safe_int(
        scene_row[
            "major_road_bottleneck_count"
        ]
    )

    flooded_major_bottleneck_count = safe_int(
        scene_row[
            "flooded_major_road_bottleneck_count"
        ]
    )

    return {
        "scene_id": str(
            scene_row[
                "scene_id"
            ]
        ),
        "physical_edge_count": (
            physical_edge_count
        ),
        "covered_edge_count": (
            covered_edge_count
        ),
        "raster_coverage_share": safe_float(
            scene_row[
                "raster_coverage_share"
            ]
        ),
        "flooded_edge_count": (
            flooded_edge_count
        ),
        "flooded_edge_share": (
            flooded_edge_share
        ),
        "low_exposure_edge_count": safe_int(
            scene_row[
                "low_flood_exposure_edge_count"
            ]
        ),
        "moderate_exposure_edge_count": safe_int(
            scene_row[
                "moderate_flood_exposure_edge_count"
            ]
        ),
        "high_exposure_edge_count": safe_int(
            scene_row[
                "high_flood_exposure_edge_count"
            ]
        ),
        "critical_edge_count": (
            critical_edge_count
        ),
        "flooded_critical_edge_count": (
            flooded_critical_edge_count
        ),
        "flooded_critical_edge_share": (
            flooded_critical_edge_share
        ),
        "low_redundancy_count": (
            low_redundancy_count
        ),
        "flooded_low_redundancy_count": (
            flooded_low_redundancy_count
        ),
        "physical_bridge_count": (
            physical_bridge_count
        ),
        "flooded_bridge_count": (
            flooded_bridge_count
        ),
        "bridge_bottleneck_count": (
            bridge_bottleneck_count
        ),
        "flooded_bridge_bottleneck_count": (
            flooded_bridge_bottleneck_count
        ),
        "major_bottleneck_count": (
            major_bottleneck_count
        ),
        "flooded_major_bottleneck_count": (
            flooded_major_bottleneck_count
        ),
        "scene_flood_burden": str(
            scene_row[
                "scene_flood_burden"
            ]
        ),
        "critical_network_disruption": str(
            scene_row[
                "critical_network_disruption"
            ]
        ),
        "disruption_score": safe_float(
            scene_row[
                "transportation_flood_disruption_score"
            ]
        ),
        "disruption_rank": safe_int(
            scene_row[
                "transportation_flood_disruption_rank"
            ]
        ),
        "grounding_reliability": str(
            scene_row[
                "grounding_reliability"
            ]
        ),
        "grounding_quality_warning": (
            normalize_optional_text(
                scene_row.get(
                    "grounding_quality_warning",
                    "",
                )
            )
        ),
        "maximum_event_flood_fraction": safe_float(
            scene_row.get(
                "maximum_event_flood_fraction_edge_evidence",
                scene_row.get(
                    "maximum_event_flood_fraction",
                    0,
                ),
            )
        ),
        "top_exposed_edges": scene_row.get(
            "top_exposed_edges",
            [],
        ),
    }


# ------------------------------------------------------------
# 4. Task-family generators
# ------------------------------------------------------------

def generate_scene_flood_assessment(
    scene_row,
    evidence,
):
    """
    Generate a scene-level roadway flood assessment.
    """

    instruction = (
        "Assess the overall roadway flood burden in this scene. "
        "Use the grounded physical-road-edge counts and flood-exposure "
        "classes, and state the assigned scene flood-burden category."
    )

    input_text = (
        f"Scene: {evidence['scene_id']}\n"
        f"Physical roadway edges: {evidence['physical_edge_count']}\n"
        f"Edges with valid raster coverage: {evidence['covered_edge_count']}\n"
        f"Flood-exposed edges: {evidence['flooded_edge_count']}\n"
        f"Flood-exposed edge share: {evidence['flooded_edge_share']:.4f}\n"
        f"Low exposure edges: {evidence['low_exposure_edge_count']}\n"
        f"Moderate exposure edges: {evidence['moderate_exposure_edge_count']}\n"
        f"High exposure edges: {evidence['high_exposure_edge_count']}"
    )

    response = (
        f"The scene has a {evidence['scene_flood_burden'].lower()} "
        f"roadway flood burden. Of "
        f"{format_count(evidence['covered_edge_count'])} roadway edges "
        f"with valid raster coverage, "
        f"{format_count(evidence['flooded_edge_count'])} "
        f"({format_share(evidence['flooded_edge_share'])}) intersect the "
        f"event-flood proxy. The exposed network includes "
        f"{format_count(evidence['low_exposure_edge_count'])} low-, "
        f"{format_count(evidence['moderate_exposure_edge_count'])} "
        f"moderate-, and "
        f"{format_count(evidence['high_exposure_edge_count'])} "
        f"high-exposure roadway edges. Therefore, the assigned scene "
        f"flood-burden class is "
        f"{evidence['scene_flood_burden']}."
    )

    return (
        instruction,
        input_text,
        clean_sentence_spacing(
            response
        ),
    )


def generate_critical_network_assessment(
    scene_row,
    evidence,
):
    """
    Generate a critical transportation-network assessment.
    """

    instruction = (
        "Evaluate whether event flooding disrupts transportation-critical "
        "roadway elements in this scene. Consider critical edges, "
        "low-redundancy edges, bridges, and identified bottlenecks."
    )

    input_text = (
        f"Scene: {evidence['scene_id']}\n"
        f"Critical transportation edges: {evidence['critical_edge_count']}\n"
        f"Flooded critical edges: {evidence['flooded_critical_edge_count']}\n"
        f"Critical low-redundancy edges: {evidence['low_redundancy_count']}\n"
        f"Flooded low-redundancy edges: "
        f"{evidence['flooded_low_redundancy_count']}\n"
        f"Physical bridges: {evidence['physical_bridge_count']}\n"
        f"Flooded physical bridges: {evidence['flooded_bridge_count']}\n"
        f"Flooded bridge bottlenecks: "
        f"{evidence['flooded_bridge_bottleneck_count']}\n"
        f"Flooded major-road bottlenecks: "
        f"{evidence['flooded_major_bottleneck_count']}"
    )

    if (
        evidence[
            "critical_network_disruption"
        ] == "None"
    ):
        interpretation = (
            "No grounded flood intersection was identified on the "
            "transportation-critical roadway elements used to define "
            "network disruption."
        )

    else:
        interpretation = (
            f"Flood exposure affects "
            f"{format_count(evidence['flooded_critical_edge_count'])} "
            f"critical transportation edges"
        )

        if evidence[
            "flooded_low_redundancy_count"
        ] > 0:
            interpretation += (
                f", including "
                f"{format_count(evidence['flooded_low_redundancy_count'])} "
                f"critical low-redundancy edges"
            )

        if evidence[
            "flooded_bridge_count"
        ] > 0:
            interpretation += (
                f" and "
                f"{format_count(evidence['flooded_bridge_count'])} "
                f"physical bridges"
            )

        interpretation += "."

    response = (
        f"The critical-network disruption category is "
        f"{evidence['critical_network_disruption']}. "
        f"{interpretation} "
        f"The scene contains "
        f"{format_count(evidence['critical_edge_count'])} critical edges, "
        f"of which "
        f"{format_count(evidence['flooded_critical_edge_count'])} "
        f"({format_share(evidence['flooded_critical_edge_share'])}) "
        f"are flood-exposed. Flooded bridge bottlenecks: "
        f"{format_count(evidence['flooded_bridge_bottleneck_count'])}; "
        f"flooded major-road bottlenecks: "
        f"{format_count(evidence['flooded_major_bottleneck_count'])}."
    )

    return (
        instruction,
        input_text,
        clean_sentence_spacing(
            response
        ),
    )


def generate_evidence_explanation(
    scene_row,
    evidence,
):
    """
    Generate an evidence-grounded explanation.
    """

    instruction = (
        "Explain why the scene received its roadway flood-burden and "
        "critical-network disruption labels. Refer only to the supplied "
        "grounding evidence."
    )

    input_text = (
        f"Scene: {evidence['scene_id']}\n"
        f"Scene flood burden: {evidence['scene_flood_burden']}\n"
        f"Critical network disruption: "
        f"{evidence['critical_network_disruption']}\n"
        f"Flooded roadway edges: {evidence['flooded_edge_count']} "
        f"of {evidence['covered_edge_count']}\n"
        f"Flooded critical edges: "
        f"{evidence['flooded_critical_edge_count']} "
        f"of {evidence['critical_edge_count']}\n"
        f"Maximum edge flood fraction: "
        f"{evidence['maximum_event_flood_fraction']:.4f}"
    )

    top_edge_text = (
        build_top_edge_evidence_text(
            evidence[
                "top_exposed_edges"
            ]
        )
    )

    response = (
        f"The {evidence['scene_flood_burden']} scene flood-burden label "
        f"is supported by "
        f"{format_count(evidence['flooded_edge_count'])} flood-exposed "
        f"roadway edges out of "
        f"{format_count(evidence['covered_edge_count'])} edges with valid "
        f"coverage, equivalent to "
        f"{format_share(evidence['flooded_edge_share'])}. "
        f"The {evidence['critical_network_disruption']} disruption label "
        f"is supported by "
        f"{format_count(evidence['flooded_critical_edge_count'])} flooded "
        f"critical edges and "
        f"{format_count(evidence['flooded_low_redundancy_count'])} flooded "
        f"critical low-redundancy edges. The maximum observed physical-edge "
        f"event-flood fraction is "
        f"{format_share(evidence['maximum_event_flood_fraction'])}. "
        f"{top_edge_text}"
    )

    return (
        instruction,
        input_text,
        clean_sentence_spacing(
            response
        ),
    )


def generate_transportation_prioritization(
    scene_row,
    evidence,
):
    """
    Generate transportation-response prioritization.
    """

    priority_label = (
        determine_priority_label(
            evidence[
                "disruption_rank"
            ],
            EXPECTED_SCENE_COUNT,
            evidence[
                "critical_network_disruption"
            ],
        )
    )

    instruction = (
        "Determine the transportation-response priority for this scene. "
        "Use the disruption score, dataset rank, roadway flood burden, "
        "and critical-network disruption evidence."
    )

    input_text = (
        f"Scene: {evidence['scene_id']}\n"
        f"Transportation-flood disruption score: "
        f"{evidence['disruption_score']:.4f}\n"
        f"Dataset disruption rank: {evidence['disruption_rank']} "
        f"of {EXPECTED_SCENE_COUNT}\n"
        f"Scene flood burden: {evidence['scene_flood_burden']}\n"
        f"Critical network disruption: "
        f"{evidence['critical_network_disruption']}\n"
        f"Flooded critical edges: "
        f"{evidence['flooded_critical_edge_count']}"
    )

    if priority_label == "High":
        action_text = (
            "This scene should receive early transportation-system review, "
            "including verification of critical links, bridge access, "
            "low-redundancy corridors, and potential detour needs."
        )

    elif priority_label == "Moderate":
        action_text = (
            "This scene should receive targeted follow-up after the "
            "highest-ranked scenes, with attention to the identified "
            "critical and flood-exposed roadway elements."
        )

    else:
        action_text = (
            "This scene may be reviewed after higher-ranked scenes, while "
            "retaining it for monitoring and validation."
        )

    response = (
        f"The transportation-response priority is {priority_label}. "
        f"The scene has a disruption score of "
        f"{evidence['disruption_score']:.4f} and ranks "
        f"{evidence['disruption_rank']} of "
        f"{EXPECTED_SCENE_COUNT} scenes. Its roadway flood burden is "
        f"{evidence['scene_flood_burden']}, and its critical-network "
        f"disruption class is "
        f"{evidence['critical_network_disruption']}. "
        f"{action_text}"
    )

    return (
        instruction,
        input_text,
        clean_sentence_spacing(
            response
        ),
    )


def generate_reliability_aware_assessment(
    scene_row,
    evidence,
):
    """
    Generate reliability-aware assessment language.
    """

    instruction = (
        "Provide a reliability-aware assessment of roadway flood and "
        "transportation disruption in this scene. Clearly separate the "
        "grounded finding from any limitation or quality warning."
    )

    input_text = (
        f"Scene: {evidence['scene_id']}\n"
        f"Scene flood burden: {evidence['scene_flood_burden']}\n"
        f"Critical network disruption: "
        f"{evidence['critical_network_disruption']}\n"
        f"Grounding reliability: {evidence['grounding_reliability']}\n"
        f"Road-raster coverage share: "
        f"{evidence['raster_coverage_share']:.4f}\n"
        f"Grounding quality warning: "
        f"{evidence['grounding_quality_warning'] or 'No warning recorded'}"
    )

    reliability_statement = (
        build_reliability_statement(
            scene_row
        )
    )

    response = (
        f"The grounded assessment identifies a "
        f"{evidence['scene_flood_burden']} roadway flood burden and "
        f"{evidence['critical_network_disruption']} critical-network "
        f"disruption. The evidence includes "
        f"{format_count(evidence['flooded_edge_count'])} flood-exposed "
        f"roadway edges and "
        f"{format_count(evidence['flooded_critical_edge_count'])} flooded "
        f"critical transportation edges. "
        f"{reliability_statement}"
    )

    return (
        instruction,
        input_text,
        clean_sentence_spacing(
            response
        ),
    )


def generate_structured_grounding_summary(
    scene_row,
    evidence,
):
    """
    Generate a compact machine-readable grounding summary.
    """

    instruction = (
        "Return a structured grounding summary for the scene. Include "
        "flood burden, critical-network disruption, exposed-road counts, "
        "disruption ranking, and grounding reliability."
    )

    input_text = (
        f"Scene: {evidence['scene_id']}\n"
        f"Combined grounding token:\n"
        f"{scene_row['combined_transportation_flood_token']}"
    )

    structured_response = {
        "scene_id": (
            evidence[
                "scene_id"
            ]
        ),
        "scene_flood_burden": (
            evidence[
                "scene_flood_burden"
            ]
        ),
        "critical_network_disruption": (
            evidence[
                "critical_network_disruption"
            ]
        ),
        "physical_edge_count": (
            evidence[
                "physical_edge_count"
            ]
        ),
        "covered_edge_count": (
            evidence[
                "covered_edge_count"
            ]
        ),
        "flooded_edge_count": (
            evidence[
                "flooded_edge_count"
            ]
        ),
        "flooded_edge_share": round(
            evidence[
                "flooded_edge_share"
            ],
            4,
        ),
        "flooded_critical_edge_count": (
            evidence[
                "flooded_critical_edge_count"
            ]
        ),
        "flooded_low_redundancy_edge_count": (
            evidence[
                "flooded_low_redundancy_count"
            ]
        ),
        "flooded_physical_bridge_count": (
            evidence[
                "flooded_bridge_count"
            ]
        ),
        "disruption_score": round(
            evidence[
                "disruption_score"
            ],
            4,
        ),
        "disruption_rank": (
            evidence[
                "disruption_rank"
            ]
        ),
        "grounding_reliability": (
            evidence[
                "grounding_reliability"
            ]
        ),
        "grounding_version": str(
            scene_row[
                "grounding_version"
            ]
        ),
    }

    response = json.dumps(
        structured_response,
        ensure_ascii=False,
        sort_keys=False,
    )

    return (
        instruction,
        input_text,
        response,
    )


TASK_GENERATORS = {
    "scene_flood_assessment": (
        generate_scene_flood_assessment
    ),
    "critical_network_assessment": (
        generate_critical_network_assessment
    ),
    "evidence_explanation": (
        generate_evidence_explanation
    ),
    "transportation_prioritization": (
        generate_transportation_prioritization
    ),
    "reliability_aware_assessment": (
        generate_reliability_aware_assessment
    ),
    "structured_grounding_summary": (
        generate_structured_grounding_summary
    ),
}


# ------------------------------------------------------------
# 5. Generate all instruction records
# ------------------------------------------------------------

instruction_records = []

for _, scene_row in (
    instruction_source_df.iterrows()
):
    scene_id = str(
        scene_row[
            "scene_id"
        ]
    )

    evidence = extract_scene_evidence(
        scene_row
    )

    for task_family in (
        SUPPORTED_TASK_FAMILIES
    ):
        generator_function = (
            TASK_GENERATORS[
                task_family
            ]
        )

        (
            instruction_text,
            input_text,
            response_text,
        ) = generator_function(
            scene_row,
            evidence,
        )

        instruction_id = (
            create_instruction_id(
                scene_id,
                task_family,
            )
        )

        instruction_record = {
            "instruction_id": (
                instruction_id
            ),
            "scene_id": (
                scene_id
            ),
            "task_family": (
                task_family
            ),
            "task_description": (
                TASK_FAMILY_DESCRIPTIONS[
                    task_family
                ]
            ),
            "instruction": (
                instruction_text
            ),
            "input": (
                input_text
            ),
            "response": (
                response_text
            ),
            "scene_flood_burden": str(
                scene_row[
                    "scene_flood_burden"
                ]
            ),
            "critical_network_disruption": str(
                scene_row[
                    "critical_network_disruption"
                ]
            ),
            "grounding_reliability": str(
                scene_row[
                    "grounding_reliability"
                ]
            ),
            "grounding_quality_warning": (
                normalize_optional_text(
                    scene_row.get(
                        "grounding_quality_warning",
                        "",
                    )
                )
            ),
            "transportation_flood_disruption_score": round(
                safe_float(
                    scene_row[
                        "transportation_flood_disruption_score"
                    ]
                ),
                4,
            ),
            "transportation_flood_disruption_rank": safe_int(
                scene_row[
                    "transportation_flood_disruption_rank"
                ]
            ),
            "combined_transportation_flood_token": str(
                scene_row[
                    "combined_transportation_flood_token"
                ]
            ),
            "source_grounding_version": str(
                scene_row[
                    "grounding_version"
                ]
            ),
            "instruction_dataset_version": (
                INSTRUCTION_DATASET_VERSION
            ),
            "generation_mode": (
                INSTRUCTION_GENERATION_MODE
            ),
            "generated_utc": (
                INSTRUCTION_GENERATED_UTC
            ),
        }

        instruction_records.append(
            instruction_record
        )


master_instruction_df = pd.DataFrame(
    instruction_records
)


# ------------------------------------------------------------
# 6. Generate chat-format records
# ------------------------------------------------------------

chat_instruction_records = []

for instruction_record in (
    instruction_records
):
    user_content = (
        instruction_record[
            "instruction"
        ]
    )

    if instruction_record[
        "input"
    ]:
        user_content += (
            "\n\nGrounding input:\n"
            + instruction_record[
                "input"
            ]
        )

    chat_record = {
        "instruction_id": (
            instruction_record[
                "instruction_id"
            ]
        ),
        "scene_id": (
            instruction_record[
                "scene_id"
            ]
        ),
        "task_family": (
            instruction_record[
                "task_family"
            ]
        ),
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a transportation resilience assistant. "
                    "Answer only from the supplied grounded road-network "
                    "and flood evidence. Do not invent roadway conditions, "
                    "flood impacts, or infrastructure attributes."
                ),
            },
            {
                "role": "user",
                "content": (
                    user_content
                ),
            },
            {
                "role": "assistant",
                "content": (
                    instruction_record[
                        "response"
                    ]
                ),
            },
        ],
        "metadata": {
            "scene_flood_burden": (
                instruction_record[
                    "scene_flood_burden"
                ]
            ),
            "critical_network_disruption": (
                instruction_record[
                    "critical_network_disruption"
                ]
            ),
            "grounding_reliability": (
                instruction_record[
                    "grounding_reliability"
                ]
            ),
            "transportation_flood_disruption_rank": (
                instruction_record[
                    "transportation_flood_disruption_rank"
                ]
            ),
            "source_grounding_version": (
                instruction_record[
                    "source_grounding_version"
                ]
            ),
            "instruction_dataset_version": (
                instruction_record[
                    "instruction_dataset_version"
                ]
            ),
        },
    }

    chat_instruction_records.append(
        chat_record
    )


# ------------------------------------------------------------
# 7. Validation checks
# ------------------------------------------------------------

task_family_counts_df = (
    master_instruction_df[
        "task_family"
    ]
    .value_counts()
    .rename_axis(
        "task_family"
    )
    .reset_index(
        name="instruction_count"
    )
)


scene_instruction_counts_df = (
    master_instruction_df
    .groupby(
        "scene_id",
        as_index=False,
    )
    .agg(
        instruction_count=(
            "instruction_id",
            "count",
        ),
        unique_task_family_count=(
            "task_family",
            "nunique",
        ),
    )
)


validation_results = {
    "instruction_count_matches_expected": (
        len(
            master_instruction_df
        )
        == EXPECTED_INSTRUCTION_COUNT
    ),
    "instruction_ids_unique": (
        master_instruction_df[
            "instruction_id"
        ].duplicated().sum()
        == 0
    ),
    "all_scenes_present": (
        master_instruction_df[
            "scene_id"
        ].nunique()
        == EXPECTED_SCENE_COUNT
    ),
    "all_task_families_present": (
        set(
            master_instruction_df[
                "task_family"
            ]
        )
        == set(
            SUPPORTED_TASK_FAMILIES
        )
    ),
    "six_records_per_scene": (
        scene_instruction_counts_df[
            "instruction_count"
        ]
        .eq(
            EXPECTED_TASK_FAMILY_COUNT
        )
        .all()
    ),
    "six_unique_tasks_per_scene": (
        scene_instruction_counts_df[
            "unique_task_family_count"
        ]
        .eq(
            EXPECTED_TASK_FAMILY_COUNT
        )
        .all()
    ),
    "no_missing_instructions": (
        master_instruction_df[
            "instruction"
        ].astype(str).str.strip().ne("").all()
    ),
    "no_missing_inputs": (
        master_instruction_df[
            "input"
        ].astype(str).str.strip().ne("").all()
    ),
    "no_missing_responses": (
        master_instruction_df[
            "response"
        ].astype(str).str.strip().ne("").all()
    ),
    "all_grounding_versions_present": (
        master_instruction_df[
            "source_grounding_version"
        ].astype(str).str.strip().ne("").all()
    ),
    "all_reliability_labels_present": (
        master_instruction_df[
            "grounding_reliability"
        ].astype(str).str.strip().ne("").all()
    ),
}


print("\nVALIDATION RESULTS")
print("-" * 80)

for (
    validation_name,
    validation_result,
) in validation_results.items():
    print(
        f"{validation_name:<40}: "
        f"{validation_result}"
    )


if not all(
    validation_results.values()
):
    failed_validations = [
        validation_name
        for (
            validation_name,
            validation_result,
        ) in validation_results.items()
        if not validation_result
    ]

    raise ValueError(
        "Instruction-generation validation failed: "
        f"{failed_validations}"
    )


# ------------------------------------------------------------
# 8. Save per-scene instruction files
# ------------------------------------------------------------

for scene_id, scene_group_df in (
    master_instruction_df.groupby(
        "scene_id"
    )
):
    safe_scene_name = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(
            scene_id
        ),
    )

    scene_csv_path = (
        SCENE_INSTRUCTION_DIR
        / f"{safe_scene_name}_instructions.csv"
    )

    scene_json_path = (
        SCENE_INSTRUCTION_DIR
        / f"{safe_scene_name}_instructions.json"
    )

    scene_group_df.to_csv(
        scene_csv_path,
        index=False,
    )

    scene_records = (
        scene_group_df.to_dict(
            orient="records"
        )
    )

    with open(
        scene_json_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            scene_records,
            file,
            indent=2,
            ensure_ascii=False,
        )


# ------------------------------------------------------------
# 9. Save master instruction datasets
# ------------------------------------------------------------

MASTER_INSTRUCTION_CSV = (
    MASTER_INSTRUCTION_DIR
    / "master_grounded_instruction_records.csv"
)

MASTER_INSTRUCTION_JSON = (
    MASTER_INSTRUCTION_DIR
    / "master_grounded_instruction_records.json"
)

MASTER_CHAT_JSON = (
    MASTER_INSTRUCTION_DIR
    / "master_grounded_instruction_chat.json"
)

MASTER_CHAT_JSONL = (
    MASTER_INSTRUCTION_DIR
    / "master_grounded_instruction_chat.jsonl"
)

TASK_FAMILY_COUNTS_CSV = (
    VALIDATION_OUTPUT_DIR
    / "instruction_task_family_counts.csv"
)

SCENE_INSTRUCTION_COUNTS_CSV = (
    VALIDATION_OUTPUT_DIR
    / "scene_instruction_counts.csv"
)

INSTRUCTION_VALIDATION_CSV = (
    VALIDATION_OUTPUT_DIR
    / "instruction_generation_validation.csv"
)


master_instruction_df.to_csv(
    MASTER_INSTRUCTION_CSV,
    index=False,
)

with open(
    MASTER_INSTRUCTION_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        instruction_records,
        file,
        indent=2,
        ensure_ascii=False,
    )

with open(
    MASTER_CHAT_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        chat_instruction_records,
        file,
        indent=2,
        ensure_ascii=False,
    )

with open(
    MASTER_CHAT_JSONL,
    "w",
    encoding="utf-8",
) as file:
    for chat_record in (
        chat_instruction_records
    ):
        file.write(
            json.dumps(
                chat_record,
                ensure_ascii=False,
            )
            + "\n"
        )


task_family_counts_df.to_csv(
    TASK_FAMILY_COUNTS_CSV,
    index=False,
)

scene_instruction_counts_df.to_csv(
    SCENE_INSTRUCTION_COUNTS_CSV,
    index=False,
)

instruction_validation_df = pd.DataFrame(
    [
        {
            "validation_check": (
                validation_name
            ),
            "passed": (
                validation_result
            ),
        }
        for (
            validation_name,
            validation_result,
        ) in validation_results.items()
    ]
)

instruction_validation_df.to_csv(
    INSTRUCTION_VALIDATION_CSV,
    index=False,
)


# ------------------------------------------------------------
# 10. Save generation summary
# ------------------------------------------------------------

instruction_generation_summary = {
    "notebook": (
        "07_instruction_generation"
    ),
    "status": (
        "instruction_records_generated"
    ),
    "instruction_dataset_version": (
        INSTRUCTION_DATASET_VERSION
    ),
    "source_grounding_version": (
        grounding_provenance.get(
            "grounding_version"
        )
    ),
    "generation_mode": (
        INSTRUCTION_GENERATION_MODE
    ),
    "generated_utc": (
        INSTRUCTION_GENERATED_UTC
    ),
    "scene_count": int(
        master_instruction_df[
            "scene_id"
        ].nunique()
    ),
    "task_family_count": int(
        master_instruction_df[
            "task_family"
        ].nunique()
    ),
    "instruction_record_count": int(
        len(
            master_instruction_df
        )
    ),
    "chat_record_count": int(
        len(
            chat_instruction_records
        )
    ),
    "high_reliability_instruction_count": int(
        (
            master_instruction_df[
                "grounding_reliability"
            ]
            == "High"
        ).sum()
    ),
    "moderate_reliability_instruction_count": int(
        (
            master_instruction_df[
                "grounding_reliability"
            ]
            == "Moderate"
        ).sum()
    ),
    "low_reliability_instruction_count": int(
        (
            master_instruction_df[
                "grounding_reliability"
            ]
            == "Low"
        ).sum()
    ),
    "supported_task_families": (
        SUPPORTED_TASK_FAMILIES
    ),
    "validation_results": {
        key: bool(value)
        for key, value in validation_results.items()
    },
    "output_files": {
        "master_instruction_csv": str(
            MASTER_INSTRUCTION_CSV
        ),
        "master_instruction_json": str(
            MASTER_INSTRUCTION_JSON
        ),
        "master_chat_json": str(
            MASTER_CHAT_JSON
        ),
        "master_chat_jsonl": str(
            MASTER_CHAT_JSONL
        ),
    },
}


INSTRUCTION_GENERATION_SUMMARY_JSON = (
    MASTER_INSTRUCTION_DIR
    / "instruction_generation_summary.json"
)

with open(
    INSTRUCTION_GENERATION_SUMMARY_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        instruction_generation_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 11. Final generation report
# ------------------------------------------------------------

print("\nTASK-FAMILY COUNTS")
print("-" * 80)

display(
    task_family_counts_df
)


print("\n" + "=" * 80)
print("GROUNDED INSTRUCTION GENERATION COMPLETE")
print("=" * 80)

print(
    f"Scenes processed              : "
    f"{master_instruction_df['scene_id'].nunique():,}"
)

print(
    f"Task families                 : "
    f"{master_instruction_df['task_family'].nunique():,}"
)

print(
    f"Instruction records generated : "
    f"{len(master_instruction_df):,}"
)

print(
    f"Chat records generated        : "
    f"{len(chat_instruction_records):,}"
)

print(
    f"Instructions per scene        : "
    f"{EXPECTED_TASK_FAMILY_COUNT:,}"
)

print(
    f"High-reliability records      : "
    f"{(master_instruction_df['grounding_reliability'] == 'High').sum():,}"
)

print(
    f"Moderate-reliability records  : "
    f"{(master_instruction_df['grounding_reliability'] == 'Moderate').sum():,}"
)

print(
    f"Low-reliability records       : "
    f"{(master_instruction_df['grounding_reliability'] == 'Low').sum():,}"
)


print("\nMASTER OUTPUT FILES")
print("-" * 80)

for output_path in [
    MASTER_INSTRUCTION_CSV,
    MASTER_INSTRUCTION_JSON,
    MASTER_CHAT_JSON,
    MASTER_CHAT_JSONL,
    TASK_FAMILY_COUNTS_CSV,
    SCENE_INSTRUCTION_COUNTS_CSV,
    INSTRUCTION_VALIDATION_CSV,
    INSTRUCTION_GENERATION_SUMMARY_JSON,
]:
    print(
        output_path
    )


print("\nINSTRUCTION PREVIEW")
print("-" * 80)

display(
    master_instruction_df[
        [
            "instruction_id",
            "scene_id",
            "task_family",
            "scene_flood_burden",
            "critical_network_disruption",
            "grounding_reliability",
            "instruction",
            "response",
        ]
    ]
    .head(
        12
    )
)

print("=" * 80)

GENERATING GROUNDED INSTRUCTION–RESPONSE RECORDS

VALIDATION RESULTS
--------------------------------------------------------------------------------
instruction_count_matches_expected      : True
instruction_ids_unique                  : True
all_scenes_present                      : True
all_task_families_present               : True
six_records_per_scene                   : True
six_unique_tasks_per_scene              : True
no_missing_instructions                 : True
no_missing_inputs                       : True
no_missing_responses                    : True
all_grounding_versions_present          : True
all_reliability_labels_present          : True



TASK-FAMILY COUNTS
--------------------------------------------------------------------------------


,task_family,instruction_count
0,scene_flood_assessment,26
1,critical_network_assessment,26
2,evidence_explanation,26
3,transportation_prioritization,26
4,reliability_aware_assessment,26
5,structured_grounding_summary,26



GROUNDED INSTRUCTION GENERATION COMPLETE
Scenes processed              : 26
Task families                 : 6
Instruction records generated : 156
Chat records generated        : 156
Instructions per scene        : 6
High-reliability records      : 114
Moderate-reliability records  : 42
Low-reliability records       : 0

MASTER OUTPUT FILES
--------------------------------------------------------------------------------
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/instruction_generation/master/master_grounded_instruction_records.csv
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/instruction_generation/master/master_grounded_instruction_records.json
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/instruction_generation/master/master_grounded_instruction_chat.json
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/instruction_generation/master/master_grounded_instruction_chat.jsonl
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/instruction_genera

,instruction_id,scene_id,task_family,scene_flood_burden,critical_network_disruption,grounding_reliability,instruction,response
0,instruction_2eaf9b2840d7,Spain_7370579,scene_flood_assessment,Moderate,High,High,Assess the overall roadway flood burden in thi...,The scene has a moderate roadway flood burden....
1,instruction_e9bc46b3c466,Spain_7370579,critical_network_assessment,Moderate,High,High,Evaluate whether event flooding disrupts trans...,The critical-network disruption category is Hi...
2,instruction_35a3d0c15aac,Spain_7370579,evidence_explanation,Moderate,High,High,Explain why the scene received its roadway flo...,The Moderate scene flood-burden label is suppo...
3,instruction_ef3330baa757,Spain_7370579,transportation_prioritization,Moderate,High,High,Determine the transportation-response priority...,The transportation-response priority is High. ...
4,instruction_e5928aff08b6,Spain_7370579,reliability_aware_assessment,Moderate,High,High,Provide a reliability-aware assessment of road...,The grounded assessment identifies a Moderate ...
5,instruction_56c2be5da28a,Spain_7370579,structured_grounding_summary,Moderate,High,High,Return a structured grounding summary for the ...,"{""scene_id"": ""Spain_7370579"", ""scene_flood_bur..."
6,instruction_294286adb877,Spain_1167260,scene_flood_assessment,Moderate,High,High,Assess the overall roadway flood burden in thi...,The scene has a moderate roadway flood burden....
7,instruction_217d950cfdd6,Spain_1167260,critical_network_assessment,Moderate,High,High,Evaluate whether event flooding disrupts trans...,The critical-network disruption category is Hi...
8,instruction_890c40b0023e,Spain_1167260,evidence_explanation,Moderate,High,High,Explain why the scene received its roadway flo...,The Moderate scene flood-burden label is suppo...
9,instruction_681106021d88,Spain_1167260,transportation_prioritization,Moderate,High,High,Determine the transportation-response priority...,The transportation-response priority is High. ...
